In [1]:
from dotenv import load_dotenv
import os
from googleapiclient.discovery import build

load_dotenv()
API_KEY = os.getenv('YOUTUBE_API_KEY')
youtube = build('youtube', 'v3', developerKey=API_KEY)

In [2]:
request = youtube.channels().list(
    part='snippet,statistics',
    forHandle='mkbhd'
)
response = request.execute()
print(response)

{'kind': 'youtube#channelListResponse', 'etag': 'pVeXzGuQy-5BRSjwfZjl2zkmYJI', 'pageInfo': {'totalResults': 1, 'resultsPerPage': 5}, 'items': [{'kind': 'youtube#channel', 'etag': 'XuIV9c4FhjoDJXoI9UxZJlKTRNs', 'id': 'UCBJycsmduvYEL83R_U4JriQ', 'snippet': {'title': 'Marques Brownlee', 'description': 'MKBHD: Quality Tech Videos | YouTuber | Geek | Consumer Electronics | Tech Head | Internet Personality!\n\nbusiness@MKBHD.com\n\nNYC', 'customUrl': '@mkbhd', 'publishedAt': '2008-03-21T15:25:54Z', 'thumbnails': {'default': {'url': 'https://yt3.ggpht.com/qu4TmIaYUlS41-dJ9gZ7DUR3nilvmB5_11i6OKSdvNnBNiyOusZP1bMN6ICnuxtjFBb6ioKgRQ=s88-c-k-c0x00ffffff-no-rj', 'width': 88, 'height': 88}, 'medium': {'url': 'https://yt3.ggpht.com/qu4TmIaYUlS41-dJ9gZ7DUR3nilvmB5_11i6OKSdvNnBNiyOusZP1bMN6ICnuxtjFBb6ioKgRQ=s240-c-k-c0x00ffffff-no-rj', 'width': 240, 'height': 240}, 'high': {'url': 'https://yt3.ggpht.com/qu4TmIaYUlS41-dJ9gZ7DUR3nilvmB5_11i6OKSdvNnBNiyOusZP1bMN6ICnuxtjFBb6ioKgRQ=s800-c-k-c0x00ffffff-no-r

In [3]:
channels = {
    'MKBHD': 'mkbhd',
    'Unbox Therapy': 'UnboxTherapy',
    'Linus Tech Tips': 'LinusTechTips',
    'Technical Guruji': 'technicalguruji',
    'Trakin Tech': 'TrakinTech',
    'Geeky Ranjit': 'geekyranjit'
}

In [4]:
channel_data = []

for name, handle in channels.items():
    request = youtube.channels().list(
        part='snippet,statistics,contentDetails',
        forHandle=handle
    )
    response = request.execute()
    
    item = response['items'][0]
    channel_data.append({
        'Channel Name': name,
        'Channel ID': item['id'],
        'Subscriber Count': item['statistics']['subscriberCount'],
        'Total Views': item['statistics']['viewCount'],
        'Video Count': item['statistics']['videoCount'],
        'Uploads Playlist ID': item['contentDetails']['relatedPlaylists']['uploads']
    })
    print(f"Fetched: {name}")

channel_data

Fetched: MKBHD
Fetched: Unbox Therapy
Fetched: Linus Tech Tips
Fetched: Technical Guruji
Fetched: Trakin Tech
Fetched: Geeky Ranjit


[{'Channel Name': 'MKBHD',
  'Channel ID': 'UCBJycsmduvYEL83R_U4JriQ',
  'Subscriber Count': '21100000',
  'Total Views': '5541785965',
  'Video Count': '1843',
  'Uploads Playlist ID': 'UUBJycsmduvYEL83R_U4JriQ'},
 {'Channel Name': 'Unbox Therapy',
  'Channel ID': 'UCsTcErHg8oDvUnTzoqsYeNw',
  'Subscriber Count': '25600000',
  'Total Views': '5188496325',
  'Video Count': '2545',
  'Uploads Playlist ID': 'UUsTcErHg8oDvUnTzoqsYeNw'},
 {'Channel Name': 'Linus Tech Tips',
  'Channel ID': 'UCXuqSBlHAE6Xw-yeJA0Tunw',
  'Subscriber Count': '16900000',
  'Total Views': '9741571245',
  'Video Count': '7892',
  'Uploads Playlist ID': 'UUXuqSBlHAE6Xw-yeJA0Tunw'},
 {'Channel Name': 'Technical Guruji',
  'Channel ID': 'UCOhHO2ICt0ti9KAh-QHvttQ',
  'Subscriber Count': '23700000',
  'Total Views': '4068119053',
  'Video Count': '6316',
  'Uploads Playlist ID': 'UUOhHO2ICt0ti9KAh-QHvttQ'},
 {'Channel Name': 'Trakin Tech',
  'Channel ID': 'UCEPL07qzVsOcHd3sMUws65g',
  'Subscriber Count': '15500000',


In [5]:
def get_video_ids(playlist_id, max_results=50):
    video_ids = []
    request = youtube.playlistItems().list(
        part='contentDetails',
        playlistId=playlist_id,
        maxResults=max_results
    )
    response = request.execute()
    for item in response['items']:
        video_ids.append(item['contentDetails']['videoId'])
    return video_ids

In [6]:
all_video_ids = []

for channel in channel_data:
    ids = get_video_ids(channel['Uploads Playlist ID'], max_results=50)
    for vid in ids:
        all_video_ids.append({'Channel Name': channel['Channel Name'], 'Video ID': vid})
    print(f"{channel['Channel Name']}: {len(ids)} videos fetched")

len(all_video_ids)

MKBHD: 50 videos fetched
Unbox Therapy: 50 videos fetched
Linus Tech Tips: 50 videos fetched
Technical Guruji: 50 videos fetched
Trakin Tech: 50 videos fetched
Geeky Ranjit: 50 videos fetched


300

In [7]:
def get_video_details(video_ids):
    all_details = []
    for i in range(0, len(video_ids), 50):
        batch = video_ids[i:i+50]
        request = youtube.videos().list(
            part='snippet,statistics,contentDetails',
            id=','.join(batch)
        )
        response = request.execute()
        for item in response['items']:
            stats = item['statistics']
            snippet = item['snippet']
            content = item['contentDetails']
            all_details.append({
                'Video ID': item['id'],
                'Channel Name': snippet.get('channelTitle'),
                'Title': snippet.get('title'),
                'Published At': snippet.get('publishedAt'),
                'Duration': content.get('duration'),
                'View Count': stats.get('viewCount', 0),
                'Like Count': stats.get('likeCount', 0),
                'Comment Count': stats.get('commentCount', 0),
                'Tags Count': len(snippet.get('tags', []))
            })
    return all_details

In [8]:
video_ids_list = [v['Video ID'] for v in all_video_ids]
video_details = get_video_details(video_ids_list)
len(video_details)

300

In [9]:
import pandas as pd

df = pd.DataFrame(video_details)
df.head()

,Video ID,Channel Name,Title,Published At,Duration,View Count,Like Count,Comment Count,Tags Count
0,ngPkbaZliaU,Marques Brownlee,The Truth About the Bezelless Concept Phone,2026-08-24T20:27:50Z,PT6M29S,963265,42606,2652,4
1,mfmdXPT7nAM,Marques Brownlee,I Said Yes to Every Email for a Month! (Again),2026-08-21T18:53:07Z,PT30M51S,2155836,75088,3615,4
2,o4SSoURPODY,Marques Brownlee,Google Pixel 11/Pro/Fold Impressions: It Is Wh...,2026-08-12T14:00:35Z,PT11M14S,3293253,86974,7090,6
3,Z6z_feacXW8,Marques Brownlee,Galaxy Z Fold 8 Review: Honeymoon's Over,2026-08-01T00:05:00Z,PT11M36S,4482831,106881,6212,0
4,_xjxwl1zLMc,Marques Brownlee,Framework 13 Pro: The Modular Laptop is Real!,2026-07-27T15:38:21Z,PT12M49S,3445950,101931,4405,6


In [10]:
df.to_csv('../raw_data/youtube_tech_channels_raw.csv', index=False)
print("Saved:", df.shape)

Saved: (300, 9)
